<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-HuggingFace-Downloader/blob/main/Colab-HuggingFace-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HuggingFace Model/Dataset Downloader

Download models, datasets, and spaces from HuggingFace Hub to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install huggingface_hub -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/HuggingFaceModelDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/HuggingFaceModelDownloader/'  # final destination

REPO_ID = ''  # Model/dataset repo ID (e.g. 'mistralai/Mistral-7B-v0.1')
REPO_TYPE = 'model'  # Repo type: 'model', 'dataset', or 'space'
FILE_PATTERN = '*'  # Glob pattern to filter files (e.g. '*.safetensors')

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def get_all_files(root):
    result = []
    for dirpath, _, filenames in os.walk(root):
        for f in filenames:
            result.append(os.path.join(dirpath, f))
    return result

def main():
    from huggingface_hub import snapshot_download
    from IPython.display import display, HTML, Javascript

    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print('Keep-alive active')

    if not REPO_ID:
        print("ERROR: Set REPO_ID in config (e.g. REPO_ID = 'mistralai/Mistral-7B-v0.1')")
        return

    begin = time.time()
    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')
    print(f'Downloading {REPO_ID}...')

    progress_display = display(HTML('<pre>Downloading...</pre>'), display_id='dl-progress')

    snapshot_download(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        local_dir=SAVE_PATH,
        allow_patterns=FILE_PATTERN,
    )

    end = time.time()
    elapsed = int(end - begin)

    items = get_all_files(SAVE_PATH)
    total_sz = sum(os.path.getsize(f) for f in items)
    print(f'Downloaded {len(items)} files ({format_bytes(total_sz)})')
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')

    if len(items) > 1:
        print(f'\nZipping {len(items)} files ({format_bytes(total_sz)})...')
        zpath = shutil.make_archive(SAVE_PATH.rstrip('/').rstrip('\\'), 'zip', SAVE_PATH)
        zsize = os.path.getsize(zpath)
        print(f'Zip: {os.path.basename(zpath)} ({format_bytes(zsize)})')
        display(HTML(f'<a href="{zpath}" download>Download ZIP</a>'))
        files.download(zpath)

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for item in os.listdir(SAVE_PATH):
        s = os.path.join(SAVE_PATH, item)
        d = os.path.join(DRIVE_PATH, item)
        if os.path.isdir(s):
            if os.path.exists(d):
                shutil.rmtree(d)
            shutil.move(s, DRIVE_PATH)
        else:
            shutil.move(s, d)
    print(f'Final: {DRIVE_PATH}')
    for item in sorted(os.listdir(DRIVE_PATH)):
        sz = os.path.getsize(os.path.join(DRIVE_PATH, item)) if os.path.isfile(os.path.join(DRIVE_PATH, item)) else 0
        print(f'  {item}' + (f' ({format_bytes(sz)})' if sz else '/'))
    print('=' * 50)

if __name__ == '__main__':
    main()